# Hypothesis 1 Testing

_If someone reaches the bottom of a story many times, they're more likely to subscribe to newsletters._

## Clarification

- "bottom" -> scroll depth & need to define what "bottom" is based on data (e.g., histogram)
- "story" -> only look at page views for articles (i.e., `page_is_article` is `True`)
- "many times" -> we're interested in cardinality, which means we should come up with a time window to not favor users who've been around longer. Let's say the time window is X days
    - First, query all users who submits a newsletter form. For each user, look at the distribution of the number of days between their first ever page view and when they submits the newsletter form to get a sense of what X should minimally be.
    - Then:
        - For users who submit a newsletter form: count number of articles they've reached bottom in past X days
        - For users who don't submit a newsletter form: get rolling counts of number of articles each reach bottom in an X-day window, then calculate the average of all those counts
- "more likely to subscribe to newsletters" -> keeping a 0/1 flag of whether a user subscribes to newsletter

## Preprocessing

In [ ]:
from enum import Enum, auto

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from helpers.pathlib import get_path_site_checkpoint
from matplotlib.ticker import PercentFormatter

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow, _StrEnum
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

In [ ]:
# INSTRUCTION: If there's a new field you'd like to add, add its name to this enum
class FieldTemp(_StrEnum):
    COUNT_BOTTOM_SCROLLS = auto()
    DAYS_FROM_EARLIEST_TO_SUBMISSION = auto()
    EARLIEST_EVENT_TSTAMP = auto()
    LAST_SUBMISSION_TSTAMP = auto()
    SUBMITTED = auto()
    USER_SCROLLS_TO_BOTTOM = auto()

In [ ]:
# Read data from CHECKPOINT 7
df_afla = pd.read_pickle(get_path_site_checkpoint(7, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(7, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(7, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(7, THE_19TH.name))

In [ ]:
# Filter to page views of articles and page views where a newsletter-form submission was recorded
def get_articles_and_submissions_dfs(df, site_name):
    df_articles = df.query(FieldNew.PAGE_IS_ARTICLE)
    df_submissions = df.query(FieldNew.FORM_SUBMIT_IS_NEWSLETTER)
    print(f"Site: {site_name}")
    print(f"Found {df_articles.shape[0]} page views for articles")
    print(f"Found {df_submissions.shape[0]} page views where a newsletter form is submitted")
    print("\n")
    return df_articles, df_submissions


df_afla_articles, df_afla_submissions = get_articles_and_submissions_dfs(df_afla, AFRO_LA.name)
df_dfp_articles, df_dfp_submissions = get_articles_and_submissions_dfs(df_dfp, DALLAS_FREE_PRESS.name)
df_ov_articles, df_ov_submissions = get_articles_and_submissions_dfs(df_ov, OPEN_VALLEJO.name)
df_19_articles, df_19_submissions = get_articles_and_submissions_dfs(df_19, THE_19TH.name)

### Define bottom threshold

In [ ]:
# Scroll depth histograms
def plot_scroll_depth_distribution(df, site_name, ax):
    df[FieldNew.SCROLL_DEPTH_MAX].plot.hist(title=f"{site_name}", bins=40, ax=ax, weights=np.ones(len(df)) / len(df))
    ax.xaxis.set_label_text("Scroll depth")
    ax.yaxis.set_label_text("Perc. of events")
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.yaxis.set_major_formatter(PercentFormatter(1))


_, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 12))
plot_scroll_depth_distribution(df_afla_articles, AFRO_LA.name, axes[0, 0])
plot_scroll_depth_distribution(df_dfp_articles, DALLAS_FREE_PRESS.name, axes[0, 1])
plot_scroll_depth_distribution(df_ov_articles, OPEN_VALLEJO.name, axes[1, 0])
plot_scroll_depth_distribution(df_19_articles, THE_19TH.name, axes[1, 1])
plt.show()

In [ ]:
# INSTRUCTION: ADD/UPDATE scroll depth threshold at which to consider as "bottom" here
class ThresholdBottom(float, Enum):
    AFRO_LA = 2 / 3
    DALLAS_FREE_PRESS = 0.8
    OPEN_VALLEJO = 0.7
    THE_19TH = 0.75

In [ ]:
# Add a boolean indicating whether a user has reached bottom in an event
def add_field_user_scroll_to_bottom(df, site_name, threshold_bottom):
    df = df.copy()
    df[FieldTemp.USER_SCROLLS_TO_BOTTOM] = df[FieldNew.SCROLL_DEPTH_MAX] >= threshold_bottom
    num_events_bottom = df[FieldTemp.USER_SCROLLS_TO_BOTTOM].sum()
    print(f"Site: {site_name}")
    print(
        f"Found {num_events_bottom} article page views ({num_events_bottom / df.shape[0]:.1%}) "
        + "in which the reader reaches the bottom."
    )
    print("\n")
    return df


df_afla_articles = add_field_user_scroll_to_bottom(df_afla_articles, AFRO_LA.name, ThresholdBottom.AFRO_LA)
df_dfp_articles = add_field_user_scroll_to_bottom(
    df_dfp_articles, DALLAS_FREE_PRESS.name, ThresholdBottom.DALLAS_FREE_PRESS
)
df_ov_articles = add_field_user_scroll_to_bottom(df_ov_articles, OPEN_VALLEJO.name, ThresholdBottom.OPEN_VALLEJO)
df_19_articles = add_field_user_scroll_to_bottom(df_19_articles, THE_19TH.name, ThresholdBottom.THE_19TH)

### Get all unique users

In [ ]:
# Group events by user and get the date of earliest event for each user
def group_events(df, site_name):
    df_grouped = (
        df.groupby(FieldSnowplow.DOMAIN_USERID)
        .aggregate({FieldSnowplow.DERIVED_TSTAMP: "min"})
        .rename(columns={FieldSnowplow.DERIVED_TSTAMP: FieldTemp.EARLIEST_EVENT_TSTAMP})
    )
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} unique readers")
    print("\n")
    return df_grouped


df_afla_users = group_events(df_afla, AFRO_LA.name)
df_dfp_users = group_events(df_dfp, DALLAS_FREE_PRESS.name)
df_ov_users = group_events(df_ov, OPEN_VALLEJO.name)
df_19_users = group_events(df_19, THE_19TH.name)

### Get unique users who sign up

In [ ]:
# Group events by user. If a user has more than one event, take the most recent one to ensure X is
# as big as it can be
def group_submissions(df_submissions, df_users, site_name):
    df_grouped = (
        df_submissions.groupby(FieldSnowplow.DOMAIN_USERID)
        .aggregate({FieldSnowplow.DERIVED_TSTAMP: "max"})
        .rename(columns={FieldSnowplow.DERIVED_TSTAMP: FieldTemp.LAST_SUBMISSION_TSTAMP})
        .merge(df_users[FieldTemp.EARLIEST_EVENT_TSTAMP], left_index=True, right_index=True)
    )
    df_grouped[FieldTemp.SUBMITTED] = 1  # True
    df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION] = (
        (df_grouped[FieldTemp.LAST_SUBMISSION_TSTAMP] - df_grouped[FieldTemp.EARLIEST_EVENT_TSTAMP]).dt.total_seconds()
        / 60
        / 60
        / 24
    )
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} readers who signed up for newsletter")
    print(
        f"Smallest window between first event and newsletter sign-up: {df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION].min():.3} days"
    )
    print(
        f"Greatest window between first event and newsletter sign-up: {df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION].max():.3} days"
    )
    print("\n")
    return df_grouped


df_afla_users_submission = group_submissions(df_afla_submissions, df_afla_users, AFRO_LA.name)
df_dfp_users_submission = group_submissions(df_dfp_submissions, df_dfp_users, DALLAS_FREE_PRESS.name)
df_ov_users_submission = group_submissions(df_ov_submissions, df_ov_users, OPEN_VALLEJO.name)
df_19_users_submission = group_submissions(df_19_submissions, df_19_users, THE_19TH.name)

### Define rolling count window

In [ ]:
class RollingWindow(int, Enum):
    AFRO_LA = 7
    DALLAS_FREE_PRESS = 7
    OPEN_VALLEJO = 7
    THE_19TH = 7

### Get unique users who read at least one article

In [ ]:
# Group article-view events by user & get number of times each user scrolls to bottom
def group_article_events(df_articles, df_users_submission, site_name, rolling_window_days):
    def count_bottom_visits(df_group):
        # User ID
        user_id = df_group[FieldSnowplow.DOMAIN_USERID].iloc[0]

        # Try to get user record in the submissions DataFrame
        try:
            user_submission_record = df_users_submission[user_id]
            count = count_bottom_visits_submitted(df_group, user_submission_record)
        except KeyError:
            count = count_bottom_visits_not_submitted(df_group)

        return count

    def count_bottom_visits_submitted(df_group, user_submission_record):
        window_end = user_submission_record[FieldTemp.LAST_SUBMISSION_TSTAMP]
        window_start = window_end - pd.Timedelta(days=rolling_window_days)

        df_window = df_group[window_start:window_end]  # Both ends are inclusive
        return df_window[FieldTemp.USER_SCROLLS_TO_BOTTOM].sum()

    def count_bottom_visits_not_submitted(df_group):
        ts_start = df_group.index[0]
        df_rolling_counts = df_group.rolling(f"{rolling_window_days}D", closed="both")[
            FieldTemp.USER_SCROLLS_TO_BOTTOM
        ].sum()[ts_start + pd.Timedelta(days=rolling_window_days) :]

        # Return average of different (rolling) X-day windows
        return df_rolling_counts.mean()

    grouped = (
        df_articles.reset_index()
        .set_index(FieldSnowplow.DERIVED_TSTAMP, drop=False)
        .sort_index()
        .groupby(FieldSnowplow.DOMAIN_USERID)
        .apply(count_bottom_visits)
        .rename(FieldTemp.COUNT_BOTTOM_SCROLLS)
    )

    if isinstance(grouped, pd.Series):
        grouped = pd.DataFrame(grouped)

    print(f"Finish grouping for site {site_name}")

    return grouped


df_afla_users_article = group_article_events(
    df_afla_articles, df_afla_users_submission, AFRO_LA.name, RollingWindow.AFRO_LA
)
df_dfp_users_article = group_article_events(
    df_dfp_articles, df_dfp_users_submission, DALLAS_FREE_PRESS.name, RollingWindow.DALLAS_FREE_PRESS
)
df_ov_users_article = group_article_events(
    df_ov_articles, df_ov_users_submission, OPEN_VALLEJO.name, RollingWindow.OPEN_VALLEJO
)
df_19_users_article = group_article_events(
    df_19_articles, df_19_users_submission, THE_19TH.name, RollingWindow.THE_19TH
)

### Get final DataFrame of users, number of article views where they reach bottom, and whether they sign up for newsletter

In [ ]:
# Table joins
def join(df_users, df_users_article, df_users_submission):
    assert {*df_users_article.index}.issubset({*df_users.index})
    assert {*df_users_submission.index}.issubset({*df_users.index})

    return (
        df_users.merge(df_users_article, how="left", left_index=True, right_index=True)
        .merge(df_users_submission, how="left", left_index=True, right_index=True)[
            [FieldTemp.COUNT_BOTTOM_SCROLLS, FieldTemp.SUBMITTED]
        ]
        .fillna(0)
    )


df_afla_final = join(df_afla_users, df_afla_users_article, df_afla_users_submission)
df_dfp_final = join(df_dfp_users, df_dfp_users_article, df_dfp_users_submission)
df_ov_final = join(df_ov_users, df_ov_users_article, df_ov_users_submission)
df_19_final = join(df_19_users, df_19_users_article, df_19_users_submission)

In [ ]:
df_afla_final.describe()

In [ ]:
df_dfp_final.describe()

In [ ]:
df_ov_final.describe()

In [ ]:
df_19_final.describe()

## Analysis